# 08 - Profiled stop, interruption, and recovery

Understand failure records offline, optionally send a standalone profiled stop, and make
an explicitly requested recovery connection.

Run cells from top to bottom in a fresh Python kernel. This notebook stands alone.
Install and launch as described in [the tutorial index](README.md).
Hardware access is disabled by default. Set your own eight-digit serial; no unit is
selected automatically. Close other notebooks' connections and other control software
before accessing the device. One connection owns the USB receive path.


In [ ]:
from dataclasses import asdict

from k10cr2 import K10CR2

SERIAL = ""  # Copy the serial from discovery and verify it on the mount's label.
CONNECT_TO_DEVICE = False  # Set True only when ready for USB access.


def configured_mount(**settings):
    # Construction validates the serial and timeouts but does not open USB.
    return K10CR2(SERIAL, **settings)

## Read a failure record offline

This deliberately constructed example does not touch USB or simulate physical stopping.
Real `MotionError.original` retains the initiating exception; `.cleanup` records the
separate stop attempt. If `confirmed_stopped` is false, physical state is unknown.


In [ ]:
from k10cr2 import CleanupResult, MotionError

example_error = MotionError(
    TimeoutError("Illustration only: motion query timed out"),
    CleanupResult(False, error="Illustration only: USB unavailable during stop"),
)
print(str(example_error))
print("Original:", repr(example_error.original))
print("Cleanup:", asdict(example_error.cleanup))

## Optional standalone profiled stop

This sends an actuator command. It is for a mount with **no other active owner**. While
`home()` or `move_absolute()` is blocking, do not use this notebook or another process
to compete for its USB connection. Interrupt that operation in its own kernel so the
owner can perform cleanup. A cooperative owner can also pass a `threading.Event` as
`cancel=`; another thread may set the event but must not call mount methods.

A notebook cell queued behind a blocking call cannot serve as an immediate stop button.
This example neither starts a background move nor induces a failure. A profiled stop
is not an emergency-stop system and controller acknowledgement is not independent
proof of mechanical rest.


In [ ]:
CONFIRM_STOP = ""  # Type "STOP " followed by the actual serial to opt in.

if CONNECT_TO_DEVICE and SERIAL and CONFIRM_STOP == f"STOP {SERIAL}":
    with configured_mount() as mount:
        try:
            result = mount.stop(timeout=5)
            print(asdict(result))
        except (Exception, KeyboardInterrupt):
            print(
                "Stop unconfirmed: PHYSICAL STATE UNKNOWN. Follow the apparatus safety procedure."
            )
            raise
else:
    print("Stop skipped; it requires connection opt-in and the exact STOP confirmation.")

## Reconnect and inspect, without replaying motion

After a query/protocol failure, or a motion failure even when cleanup confirmed a stop,
normal operations require close and explicit reconnect. Contexts in the prior notebooks
already close on exit. For an application retaining a mount object, call `mount.close()`
first; do not open a competing owner. Stop/close errors must be investigated.

Set the separate flag below to make a new connection and fresh queries. This step does
not home, enable, move, or replay anything. Reconnection alone cannot establish a new
physical home reference. Review the readback and apparatus before deciding whether an
explicit home is necessary.


In [ ]:
RECONNECT_AND_QUERY = False

if CONNECT_TO_DEVICE and RECONNECT_AND_QUERY:
    with configured_mount() as recovered:
        print("Identity:", asdict(recovered.info()))
        state = recovered.status()
        print("Status:", asdict(state), "busy:", state.busy)
        print("Position:", asdict(recovered.position()))
        print("New connection recovery_required:", recovered.recovery_required)
else:
    print("Recovery queries skipped; explicitly enable RECONNECT_AND_QUERY when ready.")

A clear software recovery flag describes the connection, not proof of a safe mechanical
state. Preserve the original error and cleanup records when reporting an incident.

Next: [09 - Bounded angle sequence](09_angle_sequence.ipynb).
